In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

#find the project root
ROOT=Path.cwd()
while not (ROOT / ".git").exists() and ROOT!=ROOT.parent:
    ROOT=ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from config import START_YEAR, END_YEAR, HOLDOUT_START, MP_THRESHOLD
import utils

df_tot=utils.player_stat_dataframe(START_YEAR, END_YEAR)
df_tot.shape

(24992, 36)

In [2]:
df_holdout=df_tot[df_tot['Year'] >= HOLDOUT_START].copy() #locked away until the very end
df_tot=df_tot[df_tot['Year'] < HOLDOUT_START].copy() #everything below uses only this
print(df_tot['Year'].max(), "last development season") #should be 2018

2018 last development season


In [3]:
#1 on every row of a player traded mid-season (his 2TM/3TM total row and each team row)
is_total=df_tot['Team'].str.endswith('TM')
df_tot['TradedMidSeason']=is_total.groupby([df_tot['Year'], df_tot['Code']]).transform('any').astype(int)

#season length = most games any single-team player played that season (50 in 1998-99, 66 in 2011-12, ...)
season_games=df_tot[~is_total].groupby('Year')['G'].max()
df_tot['SeasonG']=df_tot['Year'].map(season_games)
df_tot['GP%']=df_tot['G'] / df_tot['SeasonG']      #replaces the loader's G/82 version

#keep one row per player-season: single-team players keep their row, traded players only keep their total row
season_rows=df_tot[(df_tot['TradedMidSeason']== 0) | is_total]

#65-game rule, scaled to each season's length (65/82 = 79% of games)
data=season_rows[(season_rows['MP'] >= MP_THRESHOLD) & (season_rows['GP%'] >= 65/82)].reset_index(drop=True)

print(data.shape)
print(data.duplicated(['Year', 'Code']).sum(), "duplicate player-seasons")
print((data['MVPVS'] > 0).sum(), "vote-getter rows")
print(data.groupby('Year')['MVP'].sum().eq(1).all(), "every season still has its MVP")

(7538, 38)
0 duplicate player-seasons
610 vote-getter rows
True every season still has its MVP


In [4]:
#now feature lists and within season z-scores
feature_sets = {
    'heatmap_8':        ['PTS', 'W%', 'FT', 'eFG%', 'TRB', 'AST', 'BLK', 'STL'],
    'candidates_10':    ['PTS', 'W%', 'FT', 'eFG%', 'DRB', 'AST', 'BLK', 'STL', 'GS', '3P'],
    'rf_recall_6':      ['PTS', 'W%', 'eFG%', 'DRB', 'AST', 'BLK'],
}
#rf_recall_6 comes from the model_exploration nb which scored each subset on 2019-2025, so the results from this may be slightly more optimistic compared to the other 2
all_feats=sorted(set().union(*feature_sets.values()))

#within-season z-score for each feature: (value - season mean) / season std, then check no NaNs
for col in all_feats:
    g=data.groupby('Year')[col]
    data[col + '_z']=(data[col] - g.transform('mean'))/g.transform('std')

print(data[[c + '_z' for c in all_feats]].isna().sum())

3P_z      0
AST_z     0
BLK_z     0
DRB_z     0
FT_z      0
GS_z      0
PTS_z     0
STL_z     0
TRB_z     0
W%_z      0
eFG%_z    0
dtype: int64


### Quasi-binomial regression for MVP vote share

For each player-season, the model computes a linear score from the within-season z-scored features, then converts it to a predicted vote share between 0 and 1:
$$\text{score} = b_0 + b_1\,\text{PTS}_z + b_2\,\text{W\%}_z + \dots + b_k\,\text{STL}_z$$

$$\text{vote share} = \frac{1}{1 + e^{-\text{score}}}$$

- $b_0$ is the intercept: the score of a player at the season average in every feature.
- Each $b_i$ shows how much one standard deviation in that feature changes the score; $e^{b_i}$ is the multiplier on the odds.
- `scale='X2'` makes it quasi-binomial: the coefficients are the same as a binomial fit, but the dispersion is estimated from the data, so the standard errors and p-values are realistic.

In [5]:
fits={}
for name, feats in feature_sets.items():
    zf=[c + '_z' for c in feats]
    X=sm.add_constant(data[zf])
    fits[name]=sm.GLM(data['MVPVS'], X, family=sm.families.Binomial()).fit(scale='X2')  #quasi-binomial
    print(f"\n===== {name} (dispersion {fits[name].scale:.4f})")
    print(fits[name].summary())


===== heatmap_8 (dispersion 0.0225)
                 Generalized Linear Model Regression Results                  
Dep. Variable:                  MVPVS   No. Observations:                 7538
Model:                            GLM   Df Residuals:                     7529
Model Family:                Binomial   Df Model:                            8
Link Function:                  Logit   Scale:                        0.022520
Method:                          IRLS   Log-Likelihood:                -147.62
Date:                Wed, 04 Nov 2026   Deviance:                       105.34
Time:                        11:35:27   Pearson chi2:                     170.
No. Iterations:                    13   Pseudo R-squ. (CS):            0.08767
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        -1

In [6]:
def evaluate(df, score_col):
    hits, ranks=0, []
    for yr,g in df.groupby('Year'):
        assert g['MVP'].sum() == 1, f"{yr}: expected exactly one MVP, found {g['MVP'].sum()}"
        actual=g.loc[g['MVP'].idxmax(), 'Code']
        order=g.sort_values(score_col, ascending=False)['Code'].tolist()
        hits+=order[0] == actual #True gets 1
        ranks.append(order.index(actual) +1) #so top position is 1 and not 0
    print(f"{score_col}: picked MVP in {hits}/{len(ranks)} seasons,", f"real winner's avg rank={np.mean(ranks):.2f}")


In [7]:
import warnings
warnings.filterwarnings('ignore')   #hides repeated warnings from the fits

MIN_HISTORY=10 #seasons of training data before any predictions
years=sorted(data['Year'].unique())
test_years=years[MIN_HISTORY:] #1990 onward

for name, feats in feature_sets.items():
    zf=[c + '_z' for c in feats]
    col=f'pred_{name}'
    data[col]=np.nan

    for yr in test_years:
        train=data['Year'] < yr #only seasons before the test season
        test=data['Year'] == yr
        assert data.loc[train, 'Year'].max() < yr
        model=sm.GLM(data.loc[train, 'MVPVS'], sm.add_constant(data.loc[train, zf]),
                     family=sm.families.Binomial()).fit(scale='X2')
        data.loc[test, col]=model.predict(sm.add_constant(data.loc[test, zf], has_constant='add'))

tested=data[data['Year'].isin(test_years)]    #runs once, after all fits
for name in feature_sets:
    evaluate(tested, f'pred_{name}')
evaluate(tested, 'PTS')    #baseline, once

pred_heatmap_8: picked MVP in 17/29 seasons, real winner's avg rank=1.97
pred_candidates_10: picked MVP in 17/29 seasons, real winner's avg rank=1.97
pred_rf_recall_6: picked MVP in 18/29 seasons, real winner's avg rank=1.86
PTS: picked MVP in 10/29 seasons, real winner's avg rank=5.21


In [8]:
from sklearn.ensemble import HistGradientBoostingRegressor

for name, feats in feature_sets.items():
    for kind, cols in [('raw', feats), ('z', [c + '_z' for c in feats])]:
        col=f'gbr_{kind}_{name}'
        data[col]=np.nan
        for yr in test_years:
            train=data['Year'] < yr
            test=data['Year'] == yr
            assert data.loc[train, 'Year'].max() < yr
            model=HistGradientBoostingRegressor(random_state=12)
            model.fit(data.loc[train, cols], data.loc[train, 'MVPVS'])
            data.loc[test, col]=model.predict(data.loc[test, cols])

tested=data[data['Year'].isin(test_years)]      #rebuild so it includes the new gbr_ columns
for col in [c for c in data.columns if c.startswith(('pred_', 'gbr_'))]:
    evaluate(tested, col)


pred_heatmap_8: picked MVP in 17/29 seasons, real winner's avg rank=1.97
pred_candidates_10: picked MVP in 17/29 seasons, real winner's avg rank=1.97
pred_rf_recall_6: picked MVP in 18/29 seasons, real winner's avg rank=1.86
gbr_raw_heatmap_8: picked MVP in 24/29 seasons, real winner's avg rank=1.90
gbr_z_heatmap_8: picked MVP in 22/29 seasons, real winner's avg rank=1.66
gbr_raw_candidates_10: picked MVP in 23/29 seasons, real winner's avg rank=1.93
gbr_z_candidates_10: picked MVP in 22/29 seasons, real winner's avg rank=1.76
gbr_raw_rf_recall_6: picked MVP in 23/29 seasons, real winner's avg rank=2.00
gbr_z_rf_recall_6: picked MVP in 20/29 seasons, real winner's avg rank=1.72


In [9]:
#every MVP in the development seasons, as (season, player ID) pairs
mvp_rows=df_tot[df_tot['MVP']==1]
winners=set(zip(mvp_rows['Year'], mvp_rows['Code']))

#recency-weighted past MVPs: a win last season counts 1, two seasons ago 0.5, three seasons ago 0.25, and so on
from collections import defaultdict
wins=defaultdict(list)
for y, c in winners:
    wins[c].append(y)

data['MVP_recency']=[sum(0.5 ** (y - w - 1) for w in wins[c] if w < y)
                     for y, c in zip(data['Year'], data['Code'])]

#check: Jordan's values in 1992 and 1996
data.loc[data['Player']=='Michael Jordan', ['Year', 'MVP_recency', 'MVPVS', 'MVP']]

,Year,MVP_recency,MVPVS,MVP
703,1984,0.000000,0.172,0
1052,1986,0.000000,0.576,0
1212,1987,0.000000,0.831,1
1378,1988,1.000000,0.704,0
1569,1989,0.500000,0.613,0
1758,1990,0.250000,0.928,1
1951,1991,1.125000,0.938,1
2141,1992,1.562500,0.577,0
2715,1995,0.195312,0.986,1
2904,1996,1.097656,0.832,0


In [10]:
past=['MVP_recency'] #kept raw, not z-scored

for name, feats in feature_sets.items():
    q_cols=[c + '_z' for c in feats] + past
    g_cols=feats + past
    data[f'pred_past_{name}']=np.nan
    data[f'gbr_past_{name}']=np.nan

    for yr in test_years:
        train=data['Year'] < yr
        test=data['Year'] == yr
        assert data.loc[train, 'Year'].max() < yr
        qb=sm.GLM(data.loc[train, 'MVPVS'], sm.add_constant(data.loc[train, q_cols]),
                  family=sm.families.Binomial()).fit(scale='X2')
        data.loc[test, f'pred_past_{name}']=qb.predict(sm.add_constant(data.loc[test, q_cols], has_constant='add'))
        gb=HistGradientBoostingRegressor(random_state=12).fit(data.loc[train, g_cols], data.loc[train, 'MVPVS'])
        data.loc[test, f'gbr_past_{name}']=gb.predict(data.loc[test, g_cols])

tested=data[data['Year'].isin(test_years)]
for name in feature_sets:
    for col in [f'pred_{name}', f'pred_past_{name}', f'gbr_raw_{name}', f'gbr_past_{name}']:
        evaluate(tested, col)
    print()

pred_heatmap_8: picked MVP in 17/29 seasons, real winner's avg rank=1.97
pred_past_heatmap_8: picked MVP in 17/29 seasons, real winner's avg rank=1.97
gbr_raw_heatmap_8: picked MVP in 24/29 seasons, real winner's avg rank=1.90
gbr_past_heatmap_8: picked MVP in 22/29 seasons, real winner's avg rank=1.90

pred_candidates_10: picked MVP in 17/29 seasons, real winner's avg rank=1.97
pred_past_candidates_10: picked MVP in 17/29 seasons, real winner's avg rank=1.90
gbr_raw_candidates_10: picked MVP in 23/29 seasons, real winner's avg rank=1.93
gbr_past_candidates_10: picked MVP in 21/29 seasons, real winner's avg rank=1.83

pred_rf_recall_6: picked MVP in 18/29 seasons, real winner's avg rank=1.86
pred_past_rf_recall_6: picked MVP in 18/29 seasons, real winner's avg rank=1.86
gbr_raw_rf_recall_6: picked MVP in 23/29 seasons, real winner's avg rank=2.00
gbr_past_rf_recall_6: picked MVP in 20/29 seasons, real winner's avg rank=1.79



In [11]:
qb_all=sm.GLM(data['MVPVS'], sm.add_constant(data[[c + '_z' for c in feature_sets['candidates_10']] + ['MVP_recency']]),
              family=sm.families.Binomial()).fit(scale='X2')
print(qb_all.summary().tables[1])

                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const         -10.3240      0.133    -77.893      0.000     -10.584     -10.064
PTS_z           1.6328      0.046     35.712      0.000       1.543       1.722
W%_z            1.9761      0.040     48.861      0.000       1.897       2.055
FT_z            0.2320      0.031      7.422      0.000       0.171       0.293
eFG%_z          0.3398      0.026     12.844      0.000       0.288       0.392
DRB_z           0.4771      0.024     20.017      0.000       0.430       0.524
AST_z           0.6971      0.021     32.598      0.000       0.655       0.739
BLK_z           0.2407      0.022     11.134      0.000       0.198       0.283
STL_z           0.2441      0.022     10.990      0.000       0.201       0.288
GS_z            0.4516      0.102      4.441      0.000       0.252       0.651
3P_z           -0.0890      0.022     -4

from the last line in the table--> MVP_recency = 0.1849 , p= 0.001 --> e^0.185 ≈ 1.20, so a reigning MVP (recency = 1) has about 20% higher odds of vote share than a player with identical stats and no recent MVP